<a href="https://colab.research.google.com/github/KalinaMarkova/deep_learning_course_project/blob/main/04_Model_Training_Experiment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Grained Analysis of Propaganda in News Articles
## Notebook 04: Two-Stage Pipeline: Binary Span Detector & Technique Classifier - Experiment 2

In this notebook we will implement a Two-Model Architecture to detect propaganda in news articles. Instead of forcing a single model to solve boundary extraction and 14-class technique classification simultaneously, we divide the task into two specialized stages:

**Model A (Binary Span Detector)**: A token-classification model trained  on 3 labels (O, B-PROPAGANDA, I-PROPAGANDA) to pick up propaganda spans.

**Model B (Technique Classifier)**: A sequence-classification model that takes the isolated text spans identified by Model A and categorizes them into their specific propaganda classes (e.g., Loaded Language, Slogans, Name Calling).

By separating the tasks, each model gets to specialize and we expect this could improve the performance of the final pipeline.

In [2]:
!pip install -q transformers datasets seqeval evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.0 MB/s eta 0:00:00


In [28]:
from google.colab import drive
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification, DataCollatorWithPadding, AutoModelForSequenceClassification
import evaluate
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import glob
import os
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from tqdm.auto import tqdm
import re
import nltk
import json
from datasets import load_from_disk

nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

# Model A

Let's load the dataset and split into training, validation and test datasets.

In [6]:
drive.mount('/content/drive')

# 1. Load the flat dataset
dataset_path = '/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/exp2_span_3labels_sentence_dataset'
dataset = load_from_disk(dataset_path)

# 2. Recreate the exact same split used in previous experiments
train_temp_split = dataset.train_test_split(test_size=0.20, seed=42)
train_dataset = train_temp_split['train']
temp_dataset = train_temp_split['test']

val_test_split = temp_dataset.train_test_split(test_size=0.50, seed=42)
val_dataset = val_test_split['train']
test_dataset = val_test_split['test']

# 3. Recreate the 3-class Label Dictionaries
exp2_labels_list = ['O', 'B-PROPAGANDA', 'I-PROPAGANDA']
label2id = {label: i for i, label in enumerate(exp2_labels_list)}
id2label = {i: label for label, i in label2id.items()}

# Load RoBERTa Tokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-base", add_prefix_space=True)

print(f"Dataset split is completed.")
print(f"Train size: {len(train_dataset)}")
print(f"Validation size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Dataset split is completed.
Train size: 12022
Validation size: 1503
Test size: 1503


Let's prepare the evaluation function so the Trainer can track Precision, Recall, and F1 score at the end of each epoch.

In [7]:
# Load seqeval metric
metric = evaluate.load("seqeval")

def compute_metrics(p):
    """Calculates Precision, Recall, F1, and Accuracy ignoring -100 padding tokens."""
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Filter out -100 (special/padding tokens)
    true_predictions = [
        [exp2_labels_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [exp2_labels_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

Let's initilaize **Model A** and start training.

In [10]:
# 1. Initialize the model
model_a = AutoModelForTokenClassification.from_pretrained(
    "roberta-base",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# 2. Add attention masks (RoBERTa needs to know which tokens are real and which are padding)
def add_attention_mask(example):
    return {"attention_mask": [1] * len(example["input_ids"])}

if "attention_mask" not in train_dataset.column_names:
    print("Adding attention masks to datasets...")
    train_dataset = train_dataset.map(add_attention_mask)
    val_dataset = val_dataset.map(add_attention_mask)
    test_dataset = test_dataset.map(add_attention_mask)

# 3. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./model_a_span_detector",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

# 4. Initialize Data Collator (dynamically pads sentences to the same length)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 5. Initialize the Trainer
trainer_a = Trainer(
    model=model_a,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


# Start training
trainer_a.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.277645,0.270382,0.092879,0.104712,0.098441,0.899677
2,0.239542,0.269542,0.120773,0.130890,0.125628,0.904473
3,0.125786,0.317414,0.120936,0.162304,0.138599,0.899322


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2256, training_loss=0.22018294823719253, metrics={'train_runtime': 653.8226, 'train_samples_per_second': 55.162, 'train_steps_per_second': 3.45, 'total_flos': 1618540840813428.0, 'train_loss': 0.22018294823719253, 'epoch': 3.0})

Although Accuracy is steady at ~90%, the F1 Score peaks at only 13.85%. This happens because ~90% of all tokens in standard news text are non-propaganda (O). If a model predicts O for every single word in the dataset, it achieves 90% accuracy with an F1 score of 0. In Epoch 3, training Loss plummeted to 0.125, but Validation Loss jumped up to 0.317. This indicates that the model began memorizing the training data rather than generalizing.

In order to recify the low F1 score, we will introducing a penalty multiplier that makes missing a rare propaganda word cost the model 8 times more than missing a regular word, forcing it to actively hunt for manipulation rather than taking the easy route of ignoring it. In addition, we will reduce the number of training epochs to avoid overfitting.

In [11]:
# 1. Define Class Weights (Pushing the model to hunt for 1 and 2)
# Move weights to the same device as the model (GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
class_weights = torch.tensor([1.0, 8.0, 8.0]).to(device)

# 2. Create the Custom Trainer
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Apply the weights to the CrossEntropyLoss
        loss_fct = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)

        # Flatten predictions and labels for the loss function
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# 3. Update Training Arguments (Reduced to 2 epochs to prevent overfitting)
weighted_training_args = TrainingArguments(
    output_dir="./model_a_span_detector_weighted",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,              # <--- Reduced to 2 epochs
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

# 4. Initialize the Weighted Trainer
weighted_trainer_a = WeightedTrainer(
    model=model_a,
    args=weighted_training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,      # <--- Using the correct updated argument
    data_collator=data_collator,
    compute_metrics=compute_metrics
)



# 5. Train
weighted_trainer_a.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.224781,1.166945,0.096434,0.207679,0.131710,0.881848
2,0.180209,1.200100,0.095975,0.216405,0.132976,0.874236


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1504, training_loss=0.22277623953971457, metrics={'train_runtime': 420.721, 'train_samples_per_second': 57.15, 'train_steps_per_second': 3.575, 'total_flos': 1079655297747840.0, 'train_loss': 0.22277623953971457, 'epoch': 2.0})

The class weights worked and Recall almost doubled from 13.85% (in the previous Epoch 2) to 21.64%. This means the model found nearly twice as much actual propaganda text as it did before.

However, because we are punishing the model for missed propaganda 8x more than a regular mistake, it started guessing B-PROPAGANDA and I-PROPAGANDA much more aggressively. Precision dropped slightly to 9.59%.

The Validation loss jumped to over 1.166. This happens with weighted loss functions because when the model is wrong on an 8x-weighted token, the mathematical penalty is massive, inflating the loss number even if the core metrics (F1/Recall) are improving.



Let's initialize a fresh RoBERTa model, with the following adjustments:

1. Lowering the multiplier to 4.0 maintains a strong incentive to find rare propaganda tokens without destroying model precision.
2. Lowering the learning rate (`learning_rate=2e-5`) prevents the model from making drastic weight updates when it encounters complex, fuzzy span boundaries, helping the loss converge smoothly.
3. Restoring Epochs to 3, with gentler learning rates and softened weights, the model will learn at a controlled pace, allowing it to benefit from 3 full epochs without instantly overfitting on Epoch 3 like it did previously.
4. Add a `warmup_ratio=0.1` to force the learning rate to start at 0 and slowly climb to 2e-5 over the first 10% of the data. At the start of fine-tuning, the randomly initialized classification head produces giant, unstable gradients. Warming up the learning rate gradually over the first 10% of training protects the pre-trained RoBERTa weights from being ruined early on.



In [12]:
# 1. Start with a fresh model
model_a_improved = AutoModelForTokenClassification.from_pretrained(
    "roberta-base",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# 2. Softened Class Weights (Dialed back from 8.0 to 4.0)
device = "cuda" if torch.cuda.is_available() else "cpu"
soft_weights = torch.tensor([1.0, 4.0, 4.0]).to(device)

# 3. Create the Custom Trainer with Soft Weights
class ImprovedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Apply the softened weights
        loss_fct = nn.CrossEntropyLoss(weight=soft_weights, ignore_index=-100)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# 4. Optimized Training Arguments
improved_training_args = TrainingArguments(
    output_dir="./model_a_span_detector_optimized",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

# 5. Initialize the Improved Trainer
improved_trainer_a = ImprovedTrainer(
    model=model_a_improved,
    args=improved_training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


improved_trainer_a.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.496549,0.486683,0.040071,0.157068,0.063852,0.818434
2,0.435204,0.494319,0.072857,0.178010,0.103396,0.864644
3,0.264076,0.577899,0.087021,0.205934,0.122343,0.874466


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2256, training_loss=0.4298994375247482, metrics={'train_runtime': 778.7375, 'train_samples_per_second': 46.313, 'train_steps_per_second': 2.897, 'total_flos': 1618540840813428.0, 'train_loss': 0.4298994375247482, 'epoch': 3.0})

 Precision, Recall, and F1 score improved steadily across all three epochs, peaking at Epoch 3 (F1: 12.23%, Recall: 20.59%, Precision: 8.70%).

 The Validation Loss remained much lower and more stable (0.48 to 0.57) than the 8x run (1.18), confirming that softening the weights prevented extreme loss spikes.

 Because the penalty was cut in half, the model learned more conservatively — it took 3 full epochs to reach the exact same Recall level (20.59%) that the 8x model achieved in just 2 epochs.

 At Epoch 3, the Training Loss dropped sharply ($0.49 \rightarrow 0.26$) while the Validation Loss began creeping back up ($0.48 \rightarrow 0.57$), indicating that 3 epochs is the absolute limit before overfitting begins.

For **Model A** we would prefer the 8x Weighted Model as it gaves us higher F1 and Precision in fewer epochs. Let's evaluate the model using the competition partial metrics.

In [13]:
# 1. Get raw predictions from the 8x Weighted Model on the Test Set
print("Extracting test predictions from Model A (8x Weighted Trainer)...")
raw_preds, raw_labels, _ = weighted_trainer_a.predict(test_dataset)
pred_ids = np.argmax(raw_preds, axis=2)

# 2. Format prediction and label IDs into lists of BIO tag strings (ignoring -100 padding)
true_references = [
    [exp2_labels_list[l] for (p, l) in zip(pred, label) if l != -100]
    for pred, label in zip(pred_ids, raw_labels)
]

pred_references = [
    [exp2_labels_list[p] for (p, l) in zip(pred, label) if l != -100]
    for pred, label in zip(pred_ids, raw_labels)
]

# --- Span Extraction Function ---
def extract_spans(tags):
    """Converts a list of BIO tags into spans: (label, start_index, end_index)"""
    spans = []
    current_span = None

    for i, tag in enumerate(tags):
        if tag == 'O':
            if current_span:
                spans.append(current_span)
                current_span = None
        elif tag.startswith('B-'):
            if current_span:
                spans.append(current_span)
            current_span = (tag[2:], i, i)
        elif tag.startswith('I-'):
            if current_span and current_span[0] == tag[2:]:
                # Extend current span
                current_span = (current_span[0], current_span[1], i)
            else:
                # Malformed I-tag (starts without a B-tag)
                if current_span:
                    spans.append(current_span)
                current_span = (tag[2:], i, i)

    if current_span:
        spans.append(current_span)
    return spans

# --- Partial Overlap Score Calculation ---
total_true_spans = 0
total_pred_spans = 0
total_partial_recall_score = 0.0
total_partial_precision_score = 0.0

# Evaluate sentence by sentence
for true_tags, pred_tags in zip(true_references, pred_references):
    true_spans = extract_spans(true_tags)
    pred_spans = extract_spans(pred_tags)

    total_true_spans += len(true_spans)
    total_pred_spans += len(pred_spans)

    # 1. Calculate Partial Recall
    for t_label, t_start, t_end in true_spans:
        t_length = t_end - t_start + 1
        best_overlap = 0

        for p_label, p_start, p_end in pred_spans:
            if t_label == p_label:
                overlap_start = max(t_start, p_start)
                overlap_end = min(t_end, p_end)
                if overlap_start <= overlap_end:
                    overlap_len = overlap_end - overlap_start + 1
                    best_overlap = max(best_overlap, overlap_len)

        total_partial_recall_score += (best_overlap / t_length)

    # 2. Calculate Partial Precision
    for p_label, p_start, p_end in pred_spans:
        p_length = p_end - p_start + 1
        best_overlap = 0

        for t_label, t_start, t_end in true_spans:
            if p_label == t_label:
                overlap_start = max(p_start, t_start)
                overlap_end = min(p_end, t_end)
                if overlap_start <= overlap_end:
                    overlap_len = overlap_end - overlap_start + 1
                    best_overlap = max(best_overlap, overlap_len)

        total_partial_precision_score += (best_overlap / p_length)

# Calculate final percentages
partial_precision = total_partial_precision_score / total_pred_spans if total_pred_spans > 0 else 0
partial_recall = total_partial_recall_score / total_true_spans if total_true_spans > 0 else 0

if (partial_precision + partial_recall) > 0:
    partial_f1 = 2 * (partial_precision * partial_recall) / (partial_precision + partial_recall)
else:
    partial_f1 = 0.0

print("=" * 60)
print(" SEMEVAL-STYLE PARTIAL OVERLAP SCORES (8x Weighted Model)")
print("=" * 60)
print(f"Total Gold Spans (Ground Truth) : {total_true_spans}")
print(f"Total Predicted Spans (Model)   : {total_pred_spans}")
print("-" * 60)
print(f"Partial Precision               : {partial_precision:.4f} ({partial_precision*100:.1f}%)")
print(f"Partial Recall                  : {partial_recall:.4f} ({partial_recall*100:.1f}%)")
print(f"Partial F1 Score                : {partial_f1:.4f} ({partial_f1*100:.1f}%)")
print("=" * 60)

Extracting test predictions from Model A (8x Weighted Trainer)...


 SEMEVAL-STYLE PARTIAL OVERLAP SCORES (8x Weighted Model)
Total Gold Spans (Ground Truth) : 563
Total Predicted Spans (Model)   : 1186
------------------------------------------------------------
Partial Precision               : 0.3360 (33.6%)
Partial Recall                  : 0.6025 (60.3%)
Partial F1 Score                : 0.4315 (43.1%)


Achieving 60.3% Partial Recall on fuzzy, highly subjective text boundaries is a good result for a base RoBERTa model. It means the model is successfully capturing about 60% of the actual manipulative text in the dataset.

However, the model is predicting more than twice as many spans as actually exist to avoid the high penalty. This explains the lower Precision (33.6%). It is flagging a lot of borderline or normal text just to be safe.

For **Model A**, we want the model to over-predict rather than under-predict. If it misses a span entirely, that text is lost forever. If it accidentally flags an innocent sentence, Model B might learn to recognize it as a false positive.

Let's save the selected model.

In [ ]:
# 1. Define the target directory in your Drive
drive_save_path = '/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/model_a_span_detector_8x'

# 2. Save the model and tokenizer
weighted_trainer_a.save_model(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/model_a_span_detector_8x/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/model_a_span_detector_8x/tokenizer.json')

# Model B

For **Model B**, we are switching to Sequence Classification.  Instead of a full news article, we will feed Model B isolated, bite-sized chunks of text and Model B will read that short phrase and classify the entire sequence into one of the 14 specific propaganda techniques.
Model B does not care where the text came from or where it sat in the original document. Its only job is to become an expert at reading a phrase and identifying the psychological manipulation tactic being used.

Let's load the dataset.

In [15]:
# 1. Find the folder containing your cleaned dataset
search_path = '/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propagan*'
folder_matches = glob.glob(search_path)

if not folder_matches:
    raise FileNotFoundError(f"Could not find target folder matching: {search_path}")

target_folder = folder_matches[0]
output_csv_path = os.path.join(target_folder, 'propaganda_train_cleaned.csv')

# 2. Load the dataset into the 'df' variable
df = pd.read_csv(output_csv_path)


print(f"Total snippets loaded: {len(df)}")

# Show the first few rows just to confirm it loaded correctly
df.head(3)


Total snippets loaded: 6129


,article_id,technique,start,end,snippet,span_length,full_text_length
0,999000870,Repetition,3812,3831,migrant caravan hea,19,4597
1,111111117,Causal_Oversimplification,671,753,the delay signaled the White House was having ...,82,1064
2,780619695,Repetition,1538,1554,How inconvenient,16,6684


Because we are doing sequence classification on short text snippets, we will set `max_length=128`, which easily covers all snippet lengths. Capping the sequence length at 128 (instead of 512) trains the model incredibly fast and saves massive amounts of GPU memory.

We will then tokenize using `microsoft/deberta-v3-base`. **DeBERTa** uses "Disentangled Attention," meaning it looks at both the content of a word and its relative position in a sentence separately. It is currently one of the  best open-source models for Sequence Classification and understanding the meaning of a whole phrase.

In [16]:
# Step 1: Format Dataset for Model B

# Extract unique propaganda techniques and create label mappings
unique_techniques = df['technique'].unique().tolist()
unique_techniques.sort()

technique2id = {tech: i for i, tech in enumerate(unique_techniques)}
id2technique = {i: tech for i, tech in enumerate(unique_techniques)}

print(f"Found {len(unique_techniques)} unique techniques.")

# Create numeric 'labels' column (Plural - expected by Trainer!)
df['labels'] = df['technique'].map(technique2id)

# Clean and isolate 'text' and 'labels'
df_model_b = df[['snippet', 'labels']].copy()
df_model_b = df_model_b.rename(columns={'snippet': 'text'})
df_model_b = df_model_b.dropna(subset=['text'])

# Convert to Hugging Face Dataset and split (85% Train, 15% Val)
hf_dataset_b = Dataset.from_pandas(df_model_b)
dataset_b = hf_dataset_b.train_test_split(test_size=0.15, seed=42)

if '__index_level_0__' in dataset_b['train'].column_names:
    dataset_b = dataset_b.remove_columns(['__index_level_0__'])


# Step 2: DeBERTa-v3 Tokenization

model_b_checkpoint = "microsoft/deberta-v3-base"
print(f"Loading DeBERTa Tokenizer ({model_b_checkpoint})...")

tokenizer_b = AutoTokenizer.from_pretrained(model_b_checkpoint)

def tokenize_sequence(examples):
    return tokenizer_b(
        examples["text"],
        truncation=True,
        max_length=128,
        padding=False
    )


tokenized_dataset_b = dataset_b.map(tokenize_sequence, batched=True)

Found 14 unique techniques.
Loading DeBERTa Tokenizer (microsoft/deberta-v3-base)...


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

Map:   0%|          | 0/5209 [00:00<?, ? examples/s]

Map:   0%|          | 0/920 [00:00<?, ? examples/s]

In order to address the unbalanced data, we will compute **balanced class weights** using scikit-learn and inject them into a custom PyTorch loss function. The dynamic weights will force the model to pay much closer attention when it sees a rare class, punishing it heavily if it gets the rare classes wrong.

Standard accuracy is useless for imbalanced datasets. For evaluation, we will use **Macro averaging** which calculates the F1 score for each of the 14 classes independently, and then averages them together. This means the model's performance on a rare technique counts exactly as much as its performance on a common technique.

We will use a `learning rate of 2e-5` with warmup_steps=100. DeBERTa is very sensitive to sudden changes during fine-tuning. Forcing the learning rate to start at zero and slowly warm up over the first 100 batches prevents the model from "panicking" and destroying its pre-trained knowledge during the first epoch.

We will also use `max_grad_norm=1.0` which is a gradient clipping that acts as a limit on how drastically a model can update its weights in a single training step. If a sudden mathematical spike tries to force a massive, destabilizing update, this setting shrinks the change down to a safe maximum size of 1.0 to prevent the model from crashing.



In [17]:
# Step 3: Preparation for Training

train_labels_b = tokenized_dataset_b["train"]["labels"]
class_weights_b = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels_b),
    y=train_labels_b
)
class_weights_tensor_b = torch.tensor(class_weights_b, dtype=torch.float32).to(device)

def compute_metrics_b(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

data_collator_b = DataCollatorWithPadding(tokenizer=tokenizer_b)

# 1. Initialize Model B
model_b_safe = AutoModelForSequenceClassification.from_pretrained(
    model_b_checkpoint,
    num_labels=len(unique_techniques),
    id2label=id2technique,
    label2id=technique2id,
    torch_dtype=torch.float32  # <--- FORCE FP32 inside the model
)

# 2. Updated Training Arguments with strict FP16 disabled and gradient clipping
training_args_b_safe = TrainingArguments(
    output_dir="./model_b_technique_classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=False,
    bf16=False,
    max_grad_norm=1.0
)

# 3. Custom Trainer (Keeping the FP32 safety casts just in case)
class SequenceWeightedTrainerSafe(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        logits_fp32 = logits.to(torch.float32)
        weights_fp32 = class_weights_tensor_b.to(device=logits.device, dtype=torch.float32)

        loss_fct = nn.CrossEntropyLoss(weight=weights_fp32)
        loss = loss_fct(logits_fp32.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# 4. Initialize Trainer
trainer_b_safe = SequenceWeightedTrainerSafe(
    model=model_b_safe,
    args=training_args_b_safe,
    train_dataset=tokenized_dataset_b["train"],
    eval_dataset=tokenized_dataset_b["test"],
    processing_class=tokenizer_b,
    data_collator=data_collator_b,
    compute_metrics=compute_metrics_b
)

# Launch Training
trainer_b_safe.train()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.den

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1304, training_loss=1.5682765037004203, metrics={'train_runtime': 470.178, 'train_samples_per_second': 44.315, 'train_steps_per_second': 2.773, 'total_flos': 464517731776932.0, 'train_loss': 1.5682765037004203, 'epoch': 4.0})

Achieving 61.41% Accuracy and 52.31% Macro F1 proves that DeBERTa has learned the distinct semantic signatures of these psychological manipulation techniques.

Because we used Macro F1, rare classes contributed equally to the score. A 57.44% Recall indicates that the model is successfully identifying rare techniques rather than defaulting to Loaded Language.


Let's save now Model B.

In [ ]:
# Define save path in Google Drive
drive_save_path_b = '/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/model_b_technique_classifier_deberta'

# Save model and tokenizer
trainer_b_safe.save_model(drive_save_path_b)
tokenizer_b.save_pretrained(drive_save_path_b)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/model_b_technique_classifier_deberta/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/model_b_technique_classifier_deberta/tokenizer.json')

# Pipeline: Model A + Model B

Now we will combine Model A and Model B to run against raw validation text and measure the total pipeline score.

First we will feed the raw article text into Model A. Model A will extract the character offsets (start and end points) of any flagged text. It physically slices those text snippets out of the full document.

Then it feeds those isolated slices directly into Model B to get the final technique label. Model B recombines the boundaries and the labels into a final, clean prediction.


Now let's evaluate with the pipline using the SemEval 2020 Task 11 competition requirements.

In [29]:
# =====================================================================
# STEP 1: Load 29-Label Fine-Grained Ground Truth
# =====================================================================
print("Loading fine-grained dataset (Experiment 1)...")
exp1_path = '/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/exp1_joint_29labels_sentence_dataset'
exp1_dataset = load_from_disk(exp1_path)

# Recreate the exact same test split using seed=42
train_temp = exp1_dataset.train_test_split(test_size=0.20, seed=42)
val_test = train_temp['test'].train_test_split(test_size=0.50, seed=42)
test_exp1 = val_test['test']

# Safely extract the 29-class label names across multiple feature formats
label_names = None
label_feat = test_exp1.features['labels']

if hasattr(label_feat, 'feature') and hasattr(label_feat.feature, 'names'):
    label_names = label_feat.feature.names
elif hasattr(label_feat, 'names'):
    label_names = label_feat.names

# Fallback: Load label mapping directly from Drive if dataset features are raw integers
if not label_names:
    json_path = '/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/propaganda_tokenized_dataset/id2label.json'
    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            id2label_json = json.load(f)
            label_names = [id2label_json[str(i)] for i in range(len(id2label_json))]
    elif 'exp1_labels_list' in locals():
        label_names = exp1_labels_list
    else:
        raise ValueError("Could not locate label names. Please ensure id2label.json or exp1_labels_list exists.")

id2label_29 = {i: name for i, name in enumerate(label_names)}

eval_articles = {}
gt_records = []

for idx, sample in enumerate(test_exp1):
    art_id = str(sample.get('article_id', idx)).replace('article', '').strip()
    raw_tokens = sample.get('tokens', [])
    raw_labels = sample.get('labels', [])

    reconstructed_text = ""
    char_offset = 0
    token_spans = []

    for t in raw_tokens:
        if t in ['<s>', '</s>', '<pad>']:
            token_spans.append((char_offset, char_offset))
            continue
        cleaned_token = t.replace('Ġ', ' ')
        start_c = char_offset
        end_c = char_offset + len(cleaned_token)
        token_spans.append((start_c, end_c))
        reconstructed_text += cleaned_token
        char_offset = end_c

    eval_articles[art_id] = reconstructed_text

    # Extract fine-grained gold spans from 29-class BIO labels
    in_span = False
    current_tech = None
    span_start_char = 0
    span_end_char = 0

    for i, label_id in enumerate(raw_labels):
        if label_id == -100:
            continue

        tok_start, tok_end = token_spans[i]
        if tok_start == tok_end:
            continue

        tag_name = id2label_29.get(label_id, 'O')

        if tag_name.startswith('B-'):
            if in_span:
                gt_records.append({
                    'article_id': art_id,
                    'technique': current_tech,
                    'start': span_start_char,
                    'end': span_end_char
                })
            in_span = True
            current_tech = tag_name[2:]
            span_start_char = tok_start
            span_end_char = tok_end

        elif tag_name.startswith('I-'):
            tech_name = tag_name[2:]
            if not in_span or current_tech != tech_name:
                if in_span:
                    gt_records.append({
                        'article_id': art_id,
                        'technique': current_tech,
                        'start': span_start_char,
                        'end': span_end_char
                    })
                in_span = True
                current_tech = tech_name
                span_start_char = tok_start
            span_end_char = tok_end

        else: # 'O'
            if in_span:
                gt_records.append({
                    'article_id': art_id,
                    'technique': current_tech,
                    'start': span_start_char,
                    'end': span_end_char
                })
                in_span = False
                current_tech = None

    if in_span:
        gt_records.append({
            'article_id': art_id,
            'technique': current_tech,
            'start': span_start_char,
            'end': span_end_char
        })

val_gt_df = pd.DataFrame(gt_records)
print(f"Loaded {len(eval_articles)} test sentences and {len(val_gt_df)} fine-grained gold spans.\n")


# =====================================================================
# STEP 2: Execute Two-Stage Pipeline (Model A -> Model B)
# =====================================================================
all_pipeline_results = []
active_model_a = model_a_improved if 'model_a_improved' in locals() else model_a
active_model_b = model_b_safe if 'model_b_safe' in locals() else model_b

for art_id, article_text in tqdm(eval_articles.items(), desc="Running Full Pipeline"):
    if not article_text.strip():
        continue

    predictions = predict_two_stage_pipeline_sentence_level(
        article_text=article_text,
        model_a=active_model_a,
        tokenizer_a=tokenizer,
        model_b=active_model_b,
        tokenizer_b=tokenizer_b,
        id2technique=id2technique
    )

    for pred in predictions:
        all_pipeline_results.append({
            'article_id': str(art_id),
            'technique': pred['technique'], # Real technique from Model B
            'start': pred['start'],
            'end': pred['end'],
            'snippet': pred['snippet']
        })

pipeline_preds_df = pd.DataFrame(all_pipeline_results)
print(f"Pipeline generated {len(pipeline_preds_df)} spans with predicted techniques!")


# =====================================================================
# STEP 3: Strict SemEval Evaluation (Span Overlap + Technique Match)
# =====================================================================
def norm_tech(s):
    return str(s).replace(',', '_').replace(' ', '_').replace('-', '_').lower().strip()

gt_dict, pred_dict = {}, {}

for _, row in val_gt_df.iterrows():
    art_id = str(row['article_id'])
    if art_id not in gt_dict: gt_dict[art_id] = []
    gt_dict[art_id].append({
        'technique': norm_tech(row['technique']),
        'start': int(row['start']), 'end': int(row['end'])
    })

if not pipeline_preds_df.empty:
    for _, row in pipeline_preds_df.iterrows():
        art_id = str(row['article_id'])
        if art_id not in pred_dict: pred_dict[art_id] = []
        pred_dict[art_id].append({
            'technique': norm_tech(row['technique']),
            'start': int(row['start']), 'end': int(row['end'])
        })

total_true_spans, total_pred_spans = 0, 0
total_partial_recall, total_partial_precision = 0.0, 0.0

all_article_ids = set(gt_dict.keys()).union(set(pred_dict.keys()))

for art_id in all_article_ids:
    true_spans = gt_dict.get(art_id, [])
    pred_spans = pred_dict.get(art_id, [])

    total_true_spans += len(true_spans)
    total_pred_spans += len(pred_spans)

    # Calculate Partial Recall (Strict Technique Match)
    for t in true_spans:
        t_length = t['end'] - t['start']
        if t_length <= 0: continue
        best_overlap = 0
        for p in pred_spans:
            if t['technique'] == p['technique']:
                overlap = max(0, min(t['end'], p['end']) - max(t['start'], p['start']))
                best_overlap = max(best_overlap, overlap)
        total_partial_recall += (best_overlap / t_length)

    # Calculate Partial Precision (Strict Technique Match)
    for p in pred_spans:
        p_length = p['end'] - p['start']
        if p_length <= 0: continue
        best_overlap = 0
        for t in true_spans:
            if p['technique'] == t['technique']:
                overlap = max(0, min(p['end'], t['end']) - max(p['start'], t['start']))
                best_overlap = max(best_overlap, overlap)
        total_partial_precision += (best_overlap / p_length)

p = total_partial_precision / total_pred_spans if total_pred_spans > 0 else 0.0
r = total_partial_recall / total_true_spans if total_true_spans > 0 else 0.0
f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0

print("\n" + "=" * 60)
print("FULL TWO-STAGE PIPELINE SCORE (SPAN OVERLAP + TECHNIQUE MATCH)")
print("=" * 60)
print(f"Total Gold Spans      : {total_true_spans}")
print(f"Total Predicted Spans : {total_pred_spans}")
print(f"Partial Precision     : {p:.4f} ({p*100:.2f}%)")
print(f"Partial Recall        : {r:.4f} ({r*100:.2f}%)")
print(f"Partial F1 Score      : {f1:.4f} ({f1*100:.2f}%)")
print("=" * 60)

Loading fine-grained dataset (Experiment 1)...
Loaded 1503 test sentences and 587 fine-grained gold spans.



Running Full Pipeline:   0%|          | 0/1503 [00:00<?, ?it/s]

Pipeline generated 1021 spans with predicted techniques!

FULL TWO-STAGE PIPELINE SCORE (SPAN OVERLAP + TECHNIQUE MATCH)
Total Gold Spans      : 587
Total Predicted Spans : 1021
Partial Precision     : 0.1867 (18.67%)
Partial Recall        : 0.2979 (29.79%)
Partial F1 Score      : 0.2295 (22.95%)


The pipeline predicted 1021 spans to cover only 587 actual gold spans due to the custom ImprovedTrainer with a 4.0x class weight penalty for propaganda tokens in Model A. It got obsessed with not missing a propaganda span and predicted a lot of them. This  boosted  Recall, but it flooded Model B with false positives, effectively bringing down Precision.

Partial Recall (29.79%) is substantially higher than Partial Precision (18.67%). In imbalanced datasets, models naturally default to high precision and near-zero recall. The added class weights successfully inverted this default behavior. The pipeline is now  better at finding a portion of the real propaganda (nearly 30% partial coverage) than it is at filtering out the noise.

The final Partial F1 Score is at 22.95% as it is influenced by **cascading errors**.
